In [1]:
import polars as pl

In [2]:
df = pl.read_csv(
    "../crispr_search/plasmid_host_taxonomy_consolidated_all_possible_hosts_final_derep.tsv",
    separator="\t",
)

df = df.with_columns(
    phylum=pl.col("host").str.split(";").list.get(1),
    cclass=pl.col("host").str.split(";").list.get(2),
    order=pl.col("host").str.split(";").list.get(3),
    family=pl.col("host").str.split(";").list.get(4),
    genus=pl.col("host").str.split(";").list.get(5),
    species=pl.col("host").str.split(";").list.get(6),
)

df.head()

Plasmid,host,method,phylum,cclass,order,family,genus,species
str,str,str,str,str,str,str,str,str
"""2088090014|GPI…","""d__Bacteria;p_…","""iphop-blast""","""p__Pseudomonad…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Methylolige…","""g__Methylocean…","""s__Methylocean…"
"""2088090014|GPI…","""d__Bacteria;p_…","""iphop-blast""","""p__Pseudomonad…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Methylolige…","""g__Methylocean…",null
"""2088090014|GPI…","""d__Bacteria;p_…","""iphop-blast""","""p__Acidobacter…","""c__Vicinamibac…","""o__Vicinamibac…","""f__UBA2999""","""g__2-02-FULL-6…",null
"""2088090014|GPI…","""d__Bacteria;p_…","""iphop-blast""","""p__Acidobacter…","""c__Vicinamibac…","""o__Vicinamibac…","""f__UBA2999""","""g__2-02-FULL-6…","""s__2-02-FULL-6…"
"""2088090015|GPI…","""d__Bacteria;p_…","""iphop-blast""","""p__Actinomycet…","""c__Thermoleoph…","""o__Gaiellales""","""f__Gaiellaceae…","""g__GMQP-bins7""",null


In [3]:
isolate_host = df.filter((pl.col("Plasmid").str.contains('IMGPR|PLSDB|Refsoil')) & (pl.col('method')=='isolate')).select(pl.exclude('host','method'))

isolate_host.head()

Plasmid,phylum,cclass,order,family,genus,species
str,str,str,str,str,str,str
"""IMGPR_plasmid_…","""p__Proteobacte…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Rhizobiacea…","""g__Rhizobium""","""s__Rhizobium a…"
"""IMGPR_plasmid_…","""p__Pseudomonad…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Rhizobiacea…","""g__Rhizobium""","""s__Rhizobium a…"
"""IMGPR_plasmid_…","""p__Proteobacte…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Rhizobiacea…","""g__Rhizobium""","""s__Rhizobium a…"
"""IMGPR_plasmid_…","""p__Pseudomonad…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Rhizobiacea…","""g__Rhizobium""","""s__Rhizobium a…"
"""IMGPR_plasmid_…","""p__Pseudomonad…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Rhizobiacea…","""g__Rhizobium""","""s__Rhizobium a…"


In [4]:
df.filter(pl.col("Plasmid").str.contains('IMGPR|PLSDB|Refsoil')).group_by('Plasmid').agg(pl.col('method').n_unique())

Plasmid,method
str,u32
"""IMGPR_plasmid_…",2
"""Refsoil_NC_019…",1
"""IMGPR_plasmid_…",2
"""IMGPR_plasmid_…",2
"""IMGPR_plasmid_…",1
"""IMGPR_plasmid_…",2
"""PLSDB_NZ_CP017…",1
"""IMGPR_plasmid_…",2
"""IMGPR_plasmid_…",1


In [5]:
iphop_host = df.filter((pl.col("Plasmid").str.contains('IMGPR|PLSDB|Refsoil')) & (pl.col('method').str.contains('iphop'))).select(pl.exclude('host', 'method'))

iphop_host.head()

Plasmid,phylum,cclass,order,family,genus,species
str,str,str,str,str,str,str
"""IMGPR_plasmid_…","""p__Pseudomonad…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Rhizobiacea…","""g__Rhizobium""","""s__Rhizobium t…"
"""IMGPR_plasmid_…","""p__Pseudomonad…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Rhizobiacea…","""g__Rhizobium""","""s__Rhizobium s…"
"""IMGPR_plasmid_…","""p__Pseudomonad…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Rhizobiacea…","""g__Rhizobium""","""s__Rhizobium s…"
"""IMGPR_plasmid_…","""p__Pseudomonad…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Rhizobiacea…","""g__Rhizobium""","""s__Rhizobium c…"
"""IMGPR_plasmid_…","""p__Pseudomonad…","""c__Alphaproteo…","""o__Rhizobiales…","""f__Rhizobiacea…","""g__Rhizobium""",null


In [7]:
iphop_host.select(pl.col('Plasmid')).unique()

Plasmid
str
"""IMGPR_plasmid_…"
"""IMGPR_plasmid_…"
"""IMGPR_plasmid_…"
"""IMGPR_plasmid_…"
"""IMGPR_plasmid_…"
"""IMGPR_plasmid_…"
"""IMGPR_plasmid_…"
"""IMGPR_plasmid_…"
"""IMGPR_plasmid_…"


In [6]:
a = isolate_host.join(iphop_host, on='Plasmid', how='left').with_columns(species_agreement=pl.when(pl.col('species')==pl.col('species_right')).then(True).otherwise(False),
genus_agreement=pl.when(pl.col('genus')==pl.col('genus_right')).then(True).otherwise(False),
family_agreement=pl.when(pl.col('family')==pl.col('family_right')).then(True).otherwise(False),
order_agreement=pl.when(pl.col('order')==pl.col('order_right')).then(True).otherwise(False),
cclass_agreement=pl.when(pl.col('cclass')==pl.col('cclass_right')).then(True).otherwise(False),
phylum_agreement=pl.when(pl.col('phylum')==pl.col('phylum_right')).then(True).otherwise(False),
).filter(pl.col('phylum_right').is_not_null()).group_by('Plasmid').agg(pl.col('species_agreement').any().alias('species_agreement'), pl.col('genus_agreement').any().alias('genus_agreement'),
pl.col('family_agreement').any().alias('family_agreement'), pl.col('order_agreement').any().alias('order_agreement'),
pl.col('cclass_agreement').any().alias('cclass_agreement'), pl.col('phylum_agreement').any().alias('phylum_agreement'))

a.head()

Plasmid,species_agreement,genus_agreement,family_agreement,order_agreement,cclass_agreement,phylum_agreement
str,bool,bool,bool,bool,bool,bool
"""IMGPR_plasmid_…",false,false,true,true,true,true
"""IMGPR_plasmid_…",false,true,true,true,true,true
"""IMGPR_plasmid_…",true,true,true,true,true,true
"""IMGPR_plasmid_…",false,true,true,true,true,true
"""IMGPR_plasmid_…",true,true,true,true,true,true


In [10]:
a.select(pl.col('species_agreement').sum()/ a.shape[0], pl.col('genus_agreement').sum()/ a.shape[0], pl.col('family_agreement').sum()/ a.shape[0], pl.col('order_agreement').sum()/ a.shape[0], pl.col('cclass_agreement').sum()/ a.shape[0], pl.col('phylum_agreement').sum()/ a.shape[0])

species_agreement,genus_agreement,family_agreement,order_agreement,cclass_agreement,phylum_agreement
f64,f64,f64,f64,f64,f64
0.429327,0.909108,0.97449,0.966931,0.992819,0.780045
